# Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual <a href='https://quantra.quantinsti.com/quantra-notebook' target="_blank">here</a>.
1. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
1. Before running this notebook on your local PC:<br>
i.  You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on "**Run Codes Locally on Your Machine**" in the course.<br>
ii. You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or Python modules that might be required to run this notebook.

# Managing Portfolio Holdings with KiteConnect

While the previous notebook focused on managing active trading positions (`day` and `net`), a crucial part of portfolio management is tracking your long-term investments. These are your **holdings**, stocks you've bought for delivery (CNC) and are stored in your account.

The KiteConnect API provides a dedicated function to access this data. This notebook will cover:

1.  **Fetching Holdings Data**
2.  **Understanding Holdings Data**

## Prerequisites

* **Holdings in Your Account**: You need to have some stocks in your account to see data.

## Step 1: Authentication

As always, we start by authenticating our session.

In [1]:
import pandas as pd
from kiteconnect import KiteConnect

# A helper class to manage Kite Connect authentication.
class KiteAuth:
    def __init__(self, api_key, api_secret):
        self.api_key, self.api_secret, self.kite = api_key, api_secret, None
    
    # Method to authenticate and generate a session.
    def authenticate(self):
        try:
            kite = KiteConnect(api_key=self.api_key)
            # Generate a login URL to get the request_token.
            print("Login URL: ", kite.login_url())
            request_token = input("Enter request_token: ").strip()
            
            # Generate a session using the request_token.
            data = kite.generate_session(request_token, api_secret=self.api_secret)
            
            # Set the access_token for future API calls.
            kite.set_access_token(data['access_token'])
            self.kite = kite
            print("Authentication Successful!")
            return kite
        except Exception as e:
            print(f"Auth Error: {e}")
            return None

# --- Start Authentication ---
# IMPORTANT: Replace with your own API Key and Secret.
API_KEY = "YOUR_API_KEY"
API_SECRET = "YOUR_API_SECRET"

kite = None
if API_KEY != "YOUR_API_KEY":
    auth = KiteAuth(API_KEY, API_SECRET)
    kite = auth.authenticate()
else:
    print("Please update your API_KEY and API_SECRET to run.")

Login URL:  https://kite.zerodha.com/connect/login?api_key=hsnlxp8udalf8yeg&v=3
Enter request_token: rE6LNYqTUe3JxL97rwRzPnmtj1uOzyKx
Authentication Successful!


## Step 2: Fetching Long-Term Holdings

The `kite.holdings()` method retrieves a list of all instruments in your account. This call does not require any special authorization beyond the standard access token, as it's a fundamental part of your portfolio information.

In [2]:
if kite:
    try:
        # Fetch the list of all holdings from your account
        holdings = kite.holdings()

        # Check if any holdings data was returned
        if holdings:
            print("--- Portfolio Holdings ---")
            holdings_df = pd.DataFrame(holdings)

            # Select and display the most relevant columns
            display_cols = [
                'tradingsymbol', 'quantity', 't1_quantity',
                'average_price', 'last_price', 'pnl'
            ]
            display(holdings_df[display_cols])
        else:
            print("You have no holdings in your account.")

    except Exception as e:
        print(f"Error fetching holdings: {e}")

--- Portfolio Holdings ---


,tradingsymbol,quantity,t1_quantity,average_price,last_price,pnl
0,INFY,0,1,1584.6,1601.7,17.1


We bought 1 share of INFY yesterday, therefore, we're able to see a t1_quantity of 1 in our holdings data today.

## Step 3: Understanding the Holdings Data

The DataFrame above gives a clear snapshot of your investments. Let's break down the key columns:

- **`tradingsymbol`**: The name of the stock.
- **`quantity`**: The total number of shares you hold that are fully settled in your account.
- **`t1_quantity`**: The number of shares that you have bought but have not yet been delivered to your account (T+1 settlement cycle). These shares are not yet available for selling.
- **`average_price`**: The weighted average price at which you bought the shares in your holding.
- **`last_price`**: The last traded price of the stock.
- **`pnl`**: The total, overall Profit and Loss for this holding.

Let us now calculate and display the total invested value, current market value, and overall profit/loss for your portfolio based on these holdings.

In [3]:
if 'holdings_df' in locals() and not holdings_df.empty:
    # Calculate the total invested value
    holdings_df['invested_value'] = (holdings_df['t1_quantity'] + holdings_df['quantity']) * holdings_df['average_price']
    total_invested = holdings_df['invested_value'].sum()

    # Calculate the current market value
    holdings_df['current_value'] = (holdings_df['t1_quantity'] + holdings_df['quantity']) * holdings_df['last_price']
    total_current_value = holdings_df['current_value'].sum()

    # Calculate total portfolio P&L
    total_pnl = holdings_df['pnl'].sum()

    print(f"Total Invested Value: {total_invested:,.2f}")
    print(f"Current Market Value: {total_current_value:,.2f}")
    print(f"Total Portfolio P&L: {total_pnl:,.2f}")
else:
    print("No holdings data to analyze.")

Total Invested Value: 1,584.60
Current Market Value: 1,601.70
Total Portfolio P&L: 17.10


## Step 4: Understanding Mutual Fund Holdings Data

You can use the code kite.mf_holdings() to fetch all your mutual fund holdings as a list of dictionaries. You can convert that list into a Pandas DataFrame so you can inspect the data neatly, and (optionally) select common fields, like fund name, folio, quantity, average and last price, and P&L, for a cleaner view. 

In [2]:
# 1) Mutual funds holdings
mf = kite.mf_holdings()          # list[dict]
print(mf)

# (optional) nice tabular view
df = pd.DataFrame(mf)
print(df.head())

# Example: show essential columns if present
cols = [c for c in ["tradingsymbol","fund","folio","quantity","average_price",
                    "last_price","last_price_date","pnl"] if c in df.columns]
print(df[cols].sort_values("fund") if cols else df)

[{'tradingsymbol': 'INF204K01K49', 'fund': 'NIPPON INDIA VALUE FUND - DIRECT PLAN', 'folio': '488261605196', 'last_price': 242.7382, 'last_price_date': '', 'quantity': 163.21, 'pnl': 0.0, 'xirr': 0.0, 'average_price': 208.310828, 'discrepancy': False, 'pledged_quantity': 0.0}, {'tradingsymbol': 'INF204K01L55', 'fund': 'NIPPON INDIA ELSS TAX SAVER FUND - DIRECT PLAN', 'folio': '488261605196', 'last_price': 139.3602, 'last_price_date': '', 'quantity': 188.918, 'pnl': 0.0, 'xirr': 0.0, 'average_price': 79.395251, 'discrepancy': False, 'pledged_quantity': 0.0}, {'tradingsymbol': 'INF966L01887', 'fund': 'QUANT MID CAP FUND - DIRECT PLAN', 'folio': '51097744912', 'last_price': 230.2564, 'last_price_date': '', 'quantity': 37.314, 'pnl': 0.0, 'xirr': 0.0, 'average_price': 241.182044, 'discrepancy': False, 'pledged_quantity': 0.0}, {'tradingsymbol': 'INF200K01QX4', 'fund': 'SBI LARGE CAP FUND - DIRECT PLAN', 'folio': '29045188', 'last_price': 101.3975, 'last_price_date': '', 'quantity': 671.815

A detailed breakdown of the parameters in the holdings method is as follows:

<!-- Kite Connect — holdings() field reference (simple HTML tables) -->

<h3>Pledge / Collateral &amp; e-authorisation</h3>
<table border="1" cellpadding="6" cellspacing="0">
  <thead>
    <tr><th>Field</th><th>Meaning</th></tr>
  </thead>
  <tbody>
    <tr><td><code>collateral_quantity</code></td><td>Shares you’ve pledged to receive margin.</td></tr>
    <tr><td><code>collateral_type</code></td><td>Type/category of pledged collateral (blank if none).</td></tr>
    <tr><td><code>authorised_quantity</code></td><td>Quantity e-authorised (eDIS) for selling today.</td></tr>
    <tr><td><code>authorised_date</code></td><td>Trading day for which that e-authorisation is valid.</td></tr>
    <tr><td><code>authorisation</code></td><td>Details of the current e-authorisation session (object; may be empty).</td></tr>
  </tbody>
</table>

<h3>MTF (Margin Trading Facility)</h3>
<table border="1" cellpadding="6" cellspacing="0">
  <thead>
    <tr><th>Field</th><th>Meaning</th></tr>
  </thead>
  <tbody>
    <tr><td><code>mtf.quantity</code></td><td>Shares currently held under MTF.</td></tr>
    <tr><td><code>mtf.used_quantity</code></td><td>Part of the MTF quantity already utilised/sold.</td></tr>
    <tr><td><code>mtf.average_price</code></td><td>Average buy price for the MTF portion.</td></tr>
    <tr><td><code>mtf.value</code></td><td>Notional value of the MTF holding.</td></tr>
    <tr><td><code>mtf.initial_margin</code></td><td>Initial margin tied to the MTF holding.</td></tr>
  </tbody>
</table>

<h3>Quantities &amp; Settlement</h3>
<table border="1" cellpadding="6" cellspacing="0">
  <thead>
    <tr><th>Field</th><th>Meaning</th></tr>
  </thead>
  <tbody>
    <tr><td><code>quantity</code></td><td>Final (T+2 settled) quantity in demat.</td></tr>
    <tr><td><code>t1_quantity</code></td><td>T+1 shares (buy done, settlement pending).</td></tr>
    <tr><td><code>realised_quantity</code></td><td>Quantity already delivered into demat.</td></tr>
    <tr><td><code>used_quantity</code></td><td>Quantity sold from the current holding.</td></tr>
    <tr><td><code>opening_quantity</code></td><td>Quantity carried forward from yesterday.</td></tr>
  </tbody>
</table>

<h3>Prices &amp; P/L</h3>
<table border="1" cellpadding="6" cellspacing="0">
  <thead>
    <tr><th>Field</th><th>Meaning</th></tr>
  </thead>
  <tbody>
    <tr><td><code>average_price</code></td><td>Your average buy price for the net holding.</td></tr>
    <tr><td><code>last_price</code></td><td>Latest traded market price.</td></tr>
    <tr><td><code>close_price</code></td><td>Yesterday’s closing price.</td></tr>
    <tr><td><code>pnl</code></td><td>Total profit/loss on this stock.</td></tr>
    <tr><td><code>day_change</code></td><td>Today’s price change (absolute).</td></tr>
    <tr><td><code>day_change_percentage</code></td><td>Today’s price change (percentage).</td></tr>
  </tbody>
</table>

<h3>Instrument &amp; Misc</h3>
<table border="1" cellpadding="6" cellspacing="0">
  <thead>
    <tr><th>Field</th><th>Meaning</th></tr>
  </thead>
  <tbody>
    <tr><td><code>tradingsymbol</code></td><td>Exchange trading symbol.</td></tr>
    <tr><td><code>exchange</code></td><td>Exchange on which it’s listed.</td></tr>
    <tr><td><code>instrument_token</code></td><td>Unique instrument ID (useful for feeds).</td></tr>
    <tr><td><code>isin</code></td><td>Global ISIN for the security.</td></tr>
    <tr><td><code>product</code></td><td>Product type applied (e.g., <code>CNC</code>).</td></tr>
    <tr><td><code>discrepancy</code></td><td>True/False flag for a price discrepancy on this holding.</td></tr>
  </tbody>
</table>


## Conclusion

You now know how to distinguish between trading positions and long-term holdings and how to fetch each one appropriately. Using `kite.holdings()`, you can get a complete overview of your investment portfolio.

This allows you to build applications that not only execute trades but also track the performance of your long-term investments, calculate portfolio returns, and provide a holistic view of your financial health.
<br><br>